# 03 · Нарушение аксиомы dummy в AgentSHAP**Ключ OpenAI не нужен.** Модель локальная: Ollama + `qwen2.5:3b` (~1.9 ГБ).```bashollama serveollama pull qwen2.5:3b```Проверяемое утверждение: инструмент, который агент не вызывает, обязанполучить нулевой вклад. Не получает.Показываем три вещи:1. **Baseline недостижим.** В коде он зашит как 1.0, но `temperature=0.5`   захардкожена в `token_shap/base.py`, поэтому два прогона с одним и тем же   тулсетом не совпадают. Измеряем настоящий потолок близости $\nu < 1$.2. **Аксиома dummy нарушается.** На промптах, где не нужен ни один инструмент,   вклады не нули, и величина смещения предсказывается из $\nu$.3. **Чинится в одну строку.** `max(0, nu - similarity)` вместо   `1 - similarity`.Вся настройка агента перенесена в пакет `agent/` — модель, инструменты,загрузка TokenSHAP, атрибуция. Здесь только эксперимент.

In [ ]:
import json, sys; sys.path.insert(0, "..")from pathlib import Pathimport numpy as npimport pandas as pdfrom agent import build_agent_shap, measure_noise_floor, HARDCODED_TEMPERATUREfrom agent.attribution import (    compare, dummy_check, tool_occlusion_importance, tool_lime_importance,    shapley_from_occlusion, corrected_contributions, predicted_lime_slope,)pd.set_option("display.precision", 4)print("температура, зашитая в библиотеке:", HARDCODED_TEMPERATURE)

In [ ]:
# Клонирует TokenSHAP при первом запуске, проверяет Ollama, собирает агентаbundle = build_agent_shap()agent_shap = bundle.agent_shaptool_names = bundle.tool_namesprint("инструменты:", tool_names)

## Разминка: что на самом деле лежит в results_dfВ туториале в комментарии написано «sampling_ratio=0.0 — полный перебор,$2^n = 16$ комбинаций». Проверим: строк должно быть 16, а не 4.

In [ ]:
prompt_finance = "What's the weather in New York and how is AAPL stock doing today?"results_df, shapley_values = agent_shap.analyze(prompt=prompt_finance, sampling_ratio=0.0)print(f"строк в results_df: {len(results_df)}   (2^n было бы {2 ** len(tool_names)})")print(f"размеры коалиций:   {sorted(set(len(str(r).split('+')) for r in results_df['tools_available']))}")print()print(results_df[["tools_available", "similarity"]].to_string(index=False))

Строк $n$, а не $2^n$: библиотека перебирает только «всё, кроме одного».Это и есть причина вырождения из ноутбука `01` — и заодно повод не веритькомментариям, включая свои собственные.Сравним три метода и закрытую форму, восстановленную в `01`.

In [ ]:
occ = tool_occlusion_importance(results_df, tool_names)lime = tool_lime_importance(results_df, tool_names)table = compare(results_df, shapley_values, tool_names)print(table.to_string())slope = np.polyfit(occ.reindex(lime.index), lime, 1)[0]print(f"\nнаклон lime по occlusion: измеренный {slope:.9f}, "      f"предсказанный {predicted_lime_slope(len(tool_names)):.9f}")print(f"колонки shapley и shapley_closed_form совпадают: "      f"{np.allclose(table['shapley'], table['shapley_closed_form'], atol=1e-3)}")

## Шаг 1. Измерить noise floorBaseline в коде — единица: предполагается, что ответ на полном набореинструментов совпадает сам с собой. Но генерация стохастична, значитнастоящий потолок$$\nu = \mathbb{E}\big[\cos(\text{ответ}(N),\ \text{ответ}'(N))\big] < 1$$Разница $1 - \nu$ — постоянное положительное смещение, которое нормировкаразмазывает по всем инструментам.Стоимость: $2 \times$ `n_pairs` вызовов. На 3B-модели локально это минуты.

In [ ]:
# Если уже мерили скриптом — берём оттуда и не гоняем модель заново:#     python ../scripts/measure_noise_floor.pycached = Path("../data/noise_floor.json")if cached.exists():    d = json.loads(cached.read_text(encoding="utf-8"))    class _Cached:        nu, sd, bias = d["nu"], d["sd"], d["bias"]        n_runs, samples = d["n_runs"], d["samples"]        behavioural_noise = d.get("behavioural_noise", float("nan"))        def expected_dummy_share(self, n): return self.bias / n        def __str__(self):            return (f"noise floor  nu = {self.nu:.4f} ± {self.sd:.4f}   "                    f"({self.n_runs} прогонов, из файла)\n"                    f"смещение   1 - nu = {self.bias:.4f}")    nf = _Cached()else:    # n_runs прогонов: первый работает базлайном, остальные с ним сравниваются.    # Векторизация одна на весь корпус — ровно как в base.py, иначе число    # будет несопоставимо с колонкой similarity из analyze().    nf = measure_noise_floor(agent_shap, prompt_finance, n_runs=25, verbose=True)print(nf)print(f"\nожидаемый паразитный вклад на инструмент ~ (1-nu)/n = "      f"{nf.expected_dummy_share(len(tool_names)):.4f}")

In [ ]:
# Полезно посмотреть на разброс: если sd большая, 25 пар малоimport matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(7, 2.6))ax.hist(nf.samples, bins=12, color="#4C78A8", edgecolor="white")ax.axvline(nf.nu, color="#E45756", lw=2, label=f"nu = {nf.nu:.3f}")ax.axvline(1.0, color="#333", ls="--", lw=1.5, label="baseline в коде = 1.0")ax.set_xlabel("косинусная близость двух прогонов с полным тулсетом")ax.set_ylabel("пар")ax.legend(frameon=False)ax.spines[["top", "right"]].set_visible(False)plt.tight_layout(); plt.show()

## Шаг 2. Промпты, где не нужен ни один инструментЕсли аксиома dummy выполняется, все вклады обязаны быть нулями.

In [ ]:
NO_TOOL_PROMPTS = [    "What is the capital of France?",    "Who wrote Anna Karenina?",    "Translate 'good morning' into Spanish.",    "What does the acronym CPU stand for?",    "Name three primary colors.",]rows = []for prompt in NO_TOOL_PROMPTS:    res, shap = agent_shap.analyze(prompt=prompt, sampling_ratio=0.0)    rows.append({"prompt": prompt[:38], **{k: round(v, 4) for k, v in shap.items()},                 "сумма": round(sum(shap.values()), 4)})dummy_df = pd.DataFrame(rows).set_index("prompt")print(dummy_df.to_string())print("\nВсе значения обязаны быть нулями. Они не нули.")

Обратите внимание на колонку «сумма»: она равна единице. Эффективностьвыполняется нормировкой — и ровно она заставляет распределить единицумежду инструментами, которых агент не вызывал. Аксиома, которую мывыбрали, принуждает к неверному ответу; это та же механика, что в теоремеоб избыточности, только заметнее.И ещё: ровно один инструмент в каждой строке получает точный ноль.Это не совпадение — нормировка сдвигает минимум в ноль по построению.

In [ ]:
zeros = (dummy_df.drop(columns="сумма").abs() < 1e-9).sum(axis=1)print("сколько точных нулей в каждой строке:")print(zeros.to_string())print("\nвсегда ровно один:", bool((zeros == 1).all()))

## Шаг 3. Починка в одну строку```python# было: baseline недостижимcontribution_i = 1 - similarity(N \ {i})# стало: baseline измеренcontribution_i = max(0, nu - similarity(N \ {i}))```Для инструмента, который агент не вызывал, `similarity` держится около$\nu$, значит вклад садится в ноль.Нормировать здесь нельзя: деление почти нулевых величин на их почтинулевую сумму заново раздувает шум до единицы. Смотрим сырые вкладыпротив шумового порога $2\sigma$.

In [ ]:
prompt_notool = NO_TOOL_PROMPTS[0]res_none, shap_none = agent_shap.analyze(prompt=prompt_notool, sampling_ratio=0.0)check = dummy_check(res_none, tool_names, noise_floor=nf.nu, noise_sd=nf.sd, k=2.0)print(f"промпт: {prompt_notool!r}\n")print(check.to_string())print(f"\nаксиома dummy выполняется для всех инструментов: {bool(check['dummy держится'].all())}")

## Шаг 4. Смещение предсказуемоЕсли объяснение верное, то средний паразитный вклад по occlusion наno-tool промптах должен быть близок к $1 - \nu$, а после поправки — к нулю.Это фальсифицируемое предсказание, а не пересказ.

In [ ]:
old_means, new_means = [], []for prompt in NO_TOOL_PROMPTS:    res, _ = agent_shap.analyze(prompt=prompt, sampling_ratio=0.0)    old_means.append(tool_occlusion_importance(res, tool_names).mean())    new_means.append(corrected_contributions(res, tool_names, nf.nu, normalize=False).mean())print(f"предсказание 1 - nu                  = {nf.bias:.4f}")print(f"средний вклад, baseline = 1          = {np.mean(old_means):.4f} "      f"± {np.std(old_means):.4f}")print(f"средний вклад, baseline = nu         = {np.mean(new_means):.4f} "      f"± {np.std(new_means):.4f}")print(f"\nсмещение объяснено: "      f"{abs(np.mean(old_means) - nf.bias) < 2 * nf.sd}")

## Что из этого идёт на слайд1. Аксиома dummy нарушается системно, а не изредка: на no-tool промптах   вклады суммируются в единицу и распределяются между инструментами,   которых агент не вызывал.2. Причина не в самой идее Шепли, а в двух строках реализации: недостижимый   baseline и нормировка, принудительно обнуляющая минимум.3. Чинится тривиально — и после починки dummy выполняется по построению.4. **Но критика A этим не лечится.** Даже с починенным baseline метод   продолжает измерять достаточность, а не необходимость. Это следует   из аксиом, а не из кода, и никакой багфикс здесь не поможет.Порядок на лекции: сначала показать дефект реализации (его видно),потом объяснить, что концептуальный дефект остаётся (его не видно,и именно поэтому о нём нужно рассказывать).